# Vector stores and semantic search



In [ ]:
from sentence_transformers import SentenceTransformer
import torch
import csv

## Part I: Basic vector store implementation

In [ ]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.documents = []
        self.embeddings = None
        self.model = embedding_model

        pass

    def add_documents(self, documents: list[Document]):
        new_embeddings = self.model.encode(
            [doc.text for doc in documents],
            convert_to_tensor=True,
            normalize_embeddings=True,)

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = torch.cat([self.embeddings, new_embeddings], dim=0)

        self.documents.extend(documents)

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        embebed_query = self.model.encode(
            [query], convert_to_tensor=True,
            normalize_embeddings = True)

        scores = embebed_query @ self.embeddings.T
        sorted_scores, sorted_idx = torch.sort(scores, descending=True)
        top_scores = sorted_scores[:top_k]
        top_indices = sorted_idx[:top_k]

        return [SearchResult(score.item(), self.documents[idx]) for score, idx in zip(top_scores, top_indices)]

## Part II: Filtering by metadata

In [ ]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        pass

    def add_documents(self, documents: list[Document]):
        pass

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        pass